# Unesco Heritage Sites Many-to-One

## Actividad: 

De los datos de **whc-sites-2018-small.csv**, se deben producir tablas correctamente normalizadas según lo especificado. Una vez colocado los datos correctos en las tablas, se ejecutará lo siguiente: 

SELECT unesco.name, year, category.name, state.name, region.name, iso.name
  FROM unesco
  JOIN category ON unesco.category_id = category.id
  JOIN iso ON unesco.iso_id = iso.id
  JOIN state ON unesco.state_id = state.id
  JOIN region ON unesco.region_id = region.id
  ORDER BY region.name, unesco.name
  LIMIT 3;

In [5]:
import os
import psycopg2
import pandas as pd
from sqlalchemy import create_engine
from tabulate import tabulate

In [3]:
import warnings

warnings.filterwarnings("ignore")

In [7]:
host = "pg.pg4e.com"
port = 5432
database = "pg4e_619b3a797b"
user = "pg4e_619b3a797b"
password = "*************"

In [169]:
#estableciendo conexión 
conn = psycopg2.connect(
    host=host,
    port=port,
    database=database,
    user=user,
    password=password
)
cur = conn.cursor()
print("Conexión exitosa.")

Conexión exitosa.


In [92]:
# definición para CREAT 
def ejecutar_sql(sql):
    try:
        cur.execute(sql)
        conn.commit()
        print("Operación realizada correctamente")
        
    except Exception as e:
        print("Error:", e)
        conn.rollback()

In [40]:
ejecutar_sql("""
DROP TABLE IF EXISTS unesco_raw;

CREATE TABLE unesco_raw(
    name TEXT, 
    description TEXT, 
    justification TEXT, 
    year INTEGER,
    longitude FLOAT, 
    latitude FLOAT, 
    area_hectares FLOAT,
    category TEXT,
    category_id INTEGER,
    state TEXT, 
    state_id INTEGER,
    region TEXT,
    region_id INTEGER, 
    iso TEXT,
    iso_id INTEGER);
""")

Operación realizada correctamente


In [17]:
ejecutar_sql('''
DROP TABLE IF EXISTS category; 

CREATE TABLE category(
  id SERIAL,
  name VARCHAR(128) UNIQUE,
  PRIMARY KEY(id)
);
''')

Operación realizada correctamente


In [19]:
#definición para SELECT 
from tabulate import tabulate

def mostrar_tabla(sql):
    try:
        cur.execute(sql)

        filas = cur.fetchall()
        columnas = [desc[0] for desc in cur.description]

        print(tabulate(
            filas,
            headers=columnas,
            tablefmt="psql"
        ))

    except Exception as e:
        print("Error:", e)
        conn.rollback()

In [48]:
mostrar_tabla('''
SELECT * FROM unesco_raw;
''')

+--------+---------------+-----------------+--------+-------------+------------+-----------------+------------+---------------+---------+------------+----------+-------------+-------+----------+
| name   | description   | justification   | year   | longitude   | latitude   | area_hectares   | category   | category_id   | state   | state_id   | region   | region_id   | iso   | iso_id   |
|--------+---------------+-----------------+--------+-------------+------------+-----------------+------------+---------------+---------+------------+----------+-------------+-------+----------|
+--------+---------------+-----------------+--------+-------------+------------+-----------------+------------+---------------+---------+------------+----------+-------------+-------+----------+


In [56]:
with open("C:/Users/whc-sites-2018-small.csv", "r", encoding="utf-8") as archivo:
    cur.copy_expert(
        """
        COPY unesco_raw(
            name,
            description,
            justification,
            year,
            longitude,
            latitude,
            area_hectares,
            category,
            state,
            region,
            iso
        )
        FROM STDIN
        WITH CSV HEADER DELIMITER ','
        """,
        archivo
    )

conn.commit()

In [58]:
mostrar_tabla('''
SELECT * FROM unesco_raw LIMIT 5;
''')

+---------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------

In [29]:
def consultar_df(sql):
    try:
        df = pd.read_sql(sql, conn)
        return df

    except Exception as e:
        print("Error:", e)
        conn.rollback()

In [31]:
df = consultar_df("""
SELECT *
FROM unesco_raw
LIMIT 5;
""")

df

,name,description,justification,year,longitude,latitude,area_hectares,category,category_id,state,state_id,region,region_id,iso,iso_id
0,Cultural Landscape and Archaeological Remains ...,<p>The cultural landscape and archaeological r...,<p><em>Criterion (i):</em> The Buddha statues ...,2003,67.825250,34.846940,158.9265,Cultural,None,Afghanistan,None,Asia and the Pacific,None,af,None
1,Minaret and Archaeological Remains of Jam,"<p>The 65m-tall Minaret of Jam is a graceful, ...",<p><em>Criterion (ii):</em> The innovative arc...,2002,64.515889,34.396417,70.0000,Cultural,None,Afghanistan,None,Asia and the Pacific,None,af,None
2,Historic Centres of Berat and Gjirokastra,<p>Berat and Gjirokastra are inscribed as rare...,NaN,2005,20.133333,40.069444,58.9000,Cultural,None,Albania,None,Europe and North America,None,al,None
3,Butrint,"<p>Inhabited since prehistoric times, Butrint ...",NaN,1992,20.026111,39.751111,NaN,Cultural,None,Albania,None,Europe and North America,None,al,None
4,Al Qal'a of Beni Hammad,<p>In a mountainous site of extraordinary beau...,NaN,1980,4.786840,35.818440,150.0000,Cultural,None,Algeria,None,Arab States,None,dz,None


## Crear las tablas faltantes 

In [61]:
ejecutar_sql('''
DROP TABLE IF EXISTS iso; 

CREATE TABLE iso(
  id SERIAL,
  name VARCHAR(128) UNIQUE,
  PRIMARY KEY(id)
);

DROP TABLE IF EXISTS state; 

CREATE TABLE state(
  id SERIAL,
  name VARCHAR(128) UNIQUE,
  PRIMARY KEY(id)
);

DROP TABLE IF EXISTS region; 

CREATE TABLE region(
  id SERIAL,
  name VARCHAR(128) UNIQUE,
  PRIMARY KEY(id)
);
''')

Operación realizada correctamente


In [37]:
def insertar_datos(sql, datos=None):
    try:
        if datos is None: #dejamos como condición los datos
            cur.execute(sql)
        else:
            cur.execute(sql, datos)
            
        conn.commit()
        print("Datos insertados correctamente")
        
    except Exception as e:
        print("Error:", e)
        conn.rollback()

## Insertar datos a category

In [65]:
insertar_datos("TRUNCATE TABLE category RESTART IDENTITY CASCADE;")
insertar_datos('''
INSERT INTO category(name)
SELECT DISTINCT category
FROM unesco_raw
WHERE category IS NOT NULL
ORDER BY category;
''')

Datos insertados correctamente
Datos insertados correctamente


In [82]:
mostrar_tabla('''
SELECT * FROM category;
''')

+------+----------+
|   id | name     |
|------+----------|
|    1 | Cultural |
|    2 | Mixed    |
|    3 | Natural  |
+------+----------+


Agregar las foreign keys (de category) a unesco_raw

In [72]:
def actualizar_datos(sql, datos=None):
    try:
        if datos is None:
            cur.execute(sql) 
        else:
            cur.execute(sql, datos)
        conn.commit()
        print("Datos actualizados")
        
    except Exception as e:
        print("Error:", e)
        conn.rollback()

In [80]:
actualizar_datos('''
UPDATE unesco_raw
SET category_id = category.id
FROM category
WHERE unesco_raw.category = category.name;
''')

Datos actualizados


Verificar la normalización

In [89]:
mostrar_tabla('''
SELECT category, category_id
FROM unesco_raw
LIMIT 10;
''')

+------------+---------------+
| category   |   category_id |
|------------+---------------|
| Cultural   |             1 |
| Cultural   |             1 |
| Cultural   |             1 |
| Cultural   |             1 |
| Cultural   |             1 |
| Cultural   |             1 |
| Mixed      |             2 |
| Cultural   |             1 |
| Cultural   |             1 |
| Cultural   |             1 |
+------------+---------------+


Restringir los foreign keys 

In [98]:
ejecutar_sql('''
ALTER TABLE unesco_raw
ADD CONSTRAINT fk_category
FOREIGN KEY (category_id)
REFERENCES category(id);
''')

Operación realizada correctamente


## Insertar datos a iso

In [100]:
insertar_datos('''
INSERT INTO iso(name)
SELECT DISTINCT iso
FROM unesco_raw
WHERE iso IS NOT NULL
ORDER BY iso;
''')

Datos insertados correctamente


In [102]:
mostrar_tabla('''
SELECT * FROM iso LIMIT 6;
''')

+------+--------+
|   id | name   |
|------+--------|
|    1 | ad     |
|    2 | ae     |
|    3 | af     |
|    4 | ag     |
|    5 | al     |
|    6 | am     |
+------+--------+


Agregar las foreign keys (iso) a unesco_raw

In [111]:
actualizar_datos('''
UPDATE unesco_raw
SET iso_id = iso.id
FROM iso
WHERE unesco_raw.iso = iso.name;
''')

Datos actualizados


In [113]:
mostrar_tabla('''
SELECT iso, iso_id
FROM unesco_raw
LIMIT 10;
''')

+-------+----------+
| iso   |   iso_id |
|-------+----------|
| af    |        3 |
| af    |        3 |
| al    |        5 |
| al    |        5 |
| dz    |       43 |
| dz    |       43 |
| dz    |       43 |
| dz    |       43 |
| dz    |       43 |
| dz    |       43 |
+-------+----------+


Restringir los foreign keys

In [120]:
ejecutar_sql('''
ALTER TABLE unesco_raw
ADD CONSTRAINT fk_iso
FOREIGN KEY (iso_id)
REFERENCES iso(id);
''')

Operación realizada correctamente


## Insertar datos a state 

In [127]:
insertar_datos('''
INSERT INTO state(name)
SELECT DISTINCT state
FROM unesco_raw
WHERE state IS NOT NULL
ORDER BY state;
''')

Datos insertados correctamente


In [129]:
mostrar_tabla('''
SELECT * FROM state LIMIT 6;
''')

+------+---------------------+
|   id | name                |
|------+---------------------|
|    1 | Afghanistan         |
|    2 | Albania             |
|    3 | Algeria             |
|    4 | Andorra             |
|    5 | Angola              |
|    6 | Antigua and Barbuda |
+------+---------------------+


Agregar las foreign keys (state) a unesco_raw

In [137]:
actualizar_datos('''
UPDATE unesco_raw
SET state_id = state.id
FROM state
WHERE unesco_raw.state = state.name;
''')

Datos actualizados


In [139]:
mostrar_tabla('''
SELECT state, state_id
FROM unesco_raw
LIMIT 10;
''')

+------------------------------------------------------+------------+
| state                                                |   state_id |
|------------------------------------------------------+------------|
| Algeria                                              |          3 |
| Tunisia                                              |        147 |
| Brazil                                               |         22 |
| Bulgaria                                             |         23 |
| United Kingdom of Great Britain and Northern Ireland |        153 |
| Egypt                                                |         46 |
| Ethiopia                                             |         50 |
| Hungary                                              |         64 |
| United Kingdom of Great Britain and Northern Ireland |        153 |
| United Republic of Tanzania                          |        154 |
+------------------------------------------------------+------------+


Restringir los foreign keys

In [145]:
ejecutar_sql('''
ALTER TABLE unesco_raw
ADD CONSTRAINT fk_state
FOREIGN KEY (state_id)
REFERENCES state(id);
''')

Operación realizada correctamente


## Insertar datos a region 

In [149]:
insertar_datos('''
INSERT INTO region(name)
SELECT DISTINCT region
FROM unesco_raw
WHERE region IS NOT NULL
ORDER BY region;
''')

Datos insertados correctamente


In [151]:
mostrar_tabla('''
SELECT * FROM region LIMIT 6;
''')

+------+---------------------------------+
|   id | name                            |
|------+---------------------------------|
|    1 | Africa                          |
|    2 | Arab States                     |
|    3 | Asia and the Pacific            |
|    4 | Europe and North America        |
|    5 | Latin America and the Caribbean |
+------+---------------------------------+


Agregar las foreign keys (state) a unesco_raw

In [158]:
actualizar_datos('''
UPDATE unesco_raw
SET region_id = region.id
FROM region
WHERE unesco_raw.region = region.name;
''')

Datos actualizados


In [160]:
mostrar_tabla('''
SELECT region, region_id
FROM unesco_raw
LIMIT 10;
''')

+---------------------------------+-------------+
| region                          |   region_id |
|---------------------------------+-------------|
| Arab States                     |           2 |
| Asia and the Pacific            |           3 |
| Europe and North America        |           4 |
| Arab States                     |           2 |
| Arab States                     |           2 |
| Arab States                     |           2 |
| Arab States                     |           2 |
| Arab States                     |           2 |
| Africa                          |           1 |
| Latin America and the Caribbean |           5 |
+---------------------------------+-------------+


Restringir los foreign keys

In [167]:
ejecutar_sql('''
ALTER TABLE unesco_raw
ADD CONSTRAINT fk_region
FOREIGN KEY (region_id)
REFERENCES region(id);
''')

Operación realizada correctamente


## Crear tabla unesco

In [180]:
ejecutar_sql("DROP TABLE unesco;")
ejecutar_sql('''
CREATE TABLE unesco (
    id SERIAL PRIMARY KEY,
    name TEXT,
    year INTEGER,
    category_id INTEGER REFERENCES category(id),
    state_id INTEGER REFERENCES state(id),
    region_id INTEGER REFERENCES region(id),
    iso_id INTEGER REFERENCES iso(id)
);
''')

Operación realizada correctamente
Operación realizada correctamente


In [182]:
insertar_datos('''
INSERT INTO unesco(
    name,
    year, 
    category_id, 
    state_id, 
    region_id, 
    iso_id
)
SELECT
    name,
    year, 
    category_id, 
    state_id, 
    region_id, 
    iso_id
FROM  unesco_raw;
''')

Datos insertados correctamente


## Parte final: AUTOGRADE 

In [186]:
mostrar_tabla('''
SELECT unesco.name, year, category.name, state.name, region.name, iso.name
  FROM unesco
  JOIN category ON unesco.category_id = category.id
  JOIN iso ON unesco.iso_id = iso.id
  JOIN state ON unesco.state_id = state.id
  JOIN region ON unesco.region_id = region.id
  ORDER BY region.name, unesco.name
  LIMIT 3;
''')

+---------------------------------+--------+----------+--------------+--------+--------+
| name                            |   year | name     | name         | name   | name   |
|---------------------------------+--------+----------+--------------+--------+--------|
| Khomani Cultural Landscape      |   2017 | Cultural | South Africa | Africa | za     |
| Aapravasi Ghat                  |   2006 | Cultural | Mauritius    | Africa | mu     |
| Air and T n r  Natural Reserves |   1991 | Natural  | Niger        | Africa | ne     |
+---------------------------------+--------+----------+--------------+--------+--------+
